# Project:  **Market Basket Analysis** for an Online Retail Bussines

**Date:**  
April 2025

**Proyect Goal:**  
Analizar las compras realizadas en una tienda online durante un año, para descubrir patrones sobre qué productos suelen comprarse juntos. Estos patrones permiten a la empresa entender mejor los hábitos de sus clientes y tomar decisiones inteligentes, como ofrecer sugerencias de compra, mejorar promociones o reorganizar productos en la tienda para aumentar las ventas.  
<small>

* Ejemplos de lo que se puede lograr con este análisis:

    - **Sugerencias automáticas:** Si un cliente agrega una tetera al carrito, el sistema puede recomendarle también tazas de té.  
    - **Promociones combinadas:** Ofrecer descuentos si alguien compra papel de regalo y tarjetas de felicitación juntas.  
    - **Organización de la tienda online:** Mostrar cojines decorativos junto a sofás porque muchos clientes los compran juntos.  
    - **Campañas de email personalizadas:** Enviar ofertas de velas perfumadas a clientes que anteriormente compraron ambientadores.  
    - **Optimización del inventario:** Saber que los manteles se venden más cuando hay promociones de vajillas y preparar stock suficiente.





**Dataset:**  
Este dataset pertenece a la empresa **"Online Retail II"**, cuyo nombre no se especifica por motivos de confidencialidad, la cual se dedica a la venta de artículos de regalo para toda ocasión, y una parte importante de sus clientes son mayoristas. El dataset recopila un año completo de sus transacciones. 

<small> 

* Overview:  
    - Time Range: 1 December 2010 – 9 December 2011 (one full year)
    - Transactions: 541,909 rows
    - Features: 8
    - Business Type: Online giftware retailer (many wholesale customers)
    - Region: Primarily UK

* Typical Columns (Features):
    - InvoiceNo: Invoice number (unique ID for each transaction)
    - StockCode: Product/item code
    - Description: Item name
    - Quantity: Number of items purchased
    - InvoiceDate: Date and time of transaction
    - UnitPrice: Price per item (in GBP)
    - CustomerID: Unique ID for each customer
    - Country: Customer’s country

* This dataset is often used for:
    - Customer Segmentation (e.g., using RFM analysis)
    - Sales Analysis & Forecasting
    - Market Basket Analysis (e.g., association rules with Apriori)
    - Data Cleaning & Preprocessing Practice


* Source:  
    - https://archive.ics.uci.edu/dataset/502/online+retail+ii  
    - https://www.kaggle.com/datasets/thedevastator/online-retail-sales-and-customer-data/data

## ✔️ 0. CONFIGURACIÓN INICIAL

In [1]:
# ◯ 0. CONFIGURACIÓN INICIAL
# -------------------------------------------------------------------------------------------------

# ✓ Ruta del dataset
ruta_archivo = '/workspaces/Final_Project_MBA/data.csv' 

# ✓ Parámetros de EDA
top_n_productos = 10  # Número de productos más vendidos a graficar

# ✓ Parámetros de Apriori
min_support = 0.01    # Soporte mínimo
min_lift = 1.0        # Lift mínimo
min_confidence = 0.5  # Confianza mínima (opcional si quieres filtrar más adelante)


## ✔️ 1. IMPORTAR LIBRERIAS

In [2]:
# ◯ 1. IMPORTAR LIBRERÍAS
# ------------------------------------------------------------------

# ✓ Librerías para la manipulación de datos
import os
import numpy as np
import datetime as dt
import pandas as pd 
from IPython.display import display

# ✓ Librerías para la visualización
import matplotlib.pyplot as plt 
import seaborn as sns

# ✓ Librerías para el análisis de datos
#! pip install --upgrade pip
#! pip install mlxtend
from mlxtend.frequent_patterns import apriori, association_rules

## ✔️ 2. CARGA DE DATOS

Objective: Obtain the data from source and get a first glimpse of their properties and presentation

In [3]:
# ◯ 2. CARGA DE DATOS
# ------------------------------------------------------------------

df_raw = pd.read_csv(ruta_archivo, encoding="ISO-8859-1")       # Cargar datos originales
df_baking = df_raw.copy()                                       # Copia para el procesamiento (aplicar limpieza y procesamiento sobre df_baking)
df = df_baking.copy()                                           # Copia final para el Analisis   

# data = pd.read_csv('data.csv', encoding="ISO-8859-1")

## ✔️ 3. EDA INICIAL

In [4]:
# ◯ 3. EDA INICIAL
# -------------------------------------------------------------------------------------------------

print("\n✅ Shape inicial:", df_raw.shape)
print("\n✅ Muestra Aleatoria de Observaciones:\n")
display(df_raw.sample(10,random_state=2025))
print("\n✅ Info general:")
print(df_raw.info())
print("\n✅ Valores nulos por columna:\n", df_raw.isnull().sum().loc[lambda x: x > 0])


✅ Shape inicial: (541910, 8)

✅ Muestra Aleatoria de Observaciones:



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
411342,572216,22338,STAR DECORATION PAINTED ZINC,10,10/21/2011 12:56,0.2,"16,015.0",United Kingdom
464062,576076,22960,JAM MAKING SET WITH JARS,2,11/13/2011 16:18,4.2,"14,382.0",United Kingdom
320553,C565001,22767,TRIPLE PHOTO FRAME CORNICE,-4,8/31/2011 16:23,9.9,"17,675.0",United Kingdom
85624,543484,21080,SET/20 RED RETROSPOT PAPER NAPKINS,12,2/9/2011 10:11,0.8,"12,721.0",France
477470,577058,22371,AIRLINE BAG VINTAGE TOKYO 78,1,11/17/2011 14:29,4.2,"18,122.0",United Kingdom
139502,548325,16259,PIECE OF CAMO STATIONERY SET,108,3/30/2011 13:01,0.2,"13,509.0",United Kingdom
407182,571847,22109,FULL ENGLISH BREAKFAST PLATE,4,10/19/2011 12:39,3.8,"15,220.0",United Kingdom
410867,572183,22383,LUNCH BAG SUKI DESIGN,30,10/21/2011 10:50,1.6,"16,153.0",United Kingdom
22511,538171,21877,HOME SWEET HOME MUG,2,12/9/2010 20:01,1.2,"17,530.0",United Kingdom
133819,547799,21929,JUMBO BAG PINK VINTAGE PAISLEY,10,3/25/2011 12:48,1.9,"17,139.0",United Kingdom



✅ Info general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      541910 non-null  object 
 1   StockCode    541910 non-null  object 
 2   Description  540456 non-null  object 
 3   Quantity     541910 non-null  int64  
 4   InvoiceDate  541910 non-null  object 
 5   Price        541910 non-null  float64
 6   Customer ID  406830 non-null  float64
 7   Country      541910 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None

✅ Valores nulos por columna:
 Description      1454
Customer ID    135080
dtype: int64


## ✔️ 4. PROCESASMIENTO DE DATOS

Objectives: Perform the data cleaning, data transformation and data reduction steps to avoid data mistmatching, noisy data or data not wrangled


In [5]:
# ◯ 4. PROCESAMIENTO DE DATOS
# -------------------------------------------------------------------------------------------------

# ✓ Eliminar filas con valores nulos en columnas
df_baking = df_baking.dropna()  

# ✓ (Opcional) Si no se usa Customer ID, puede omitirse o eliminarse
# df = df.drop(columns=['Customer ID'])


# ✓ Renombrar columnas pasandolas a minuscula
df_baking.columns = df_baking.columns.str.lower()

# ✓ Filtrar United Kingdom
#    El dataset contiene transacciones de varios países, pero nos enfocaremos en el Reino Unido,
#    ya que es donde se encuentran la mayoria de transaccciones (91.5%). 
df_baking = df_baking[df_baking['country'] == 'United Kingdom'].copy()

# ✓ Limpieza de descripciones: Normalizar y limpiar los nombres de productos
#                              Convertimos los nombres de productos (Description) a minúsculas y eliminamos espacios extra.
df_baking['description'] = df_baking['description'].str.strip().str.lower()


# ✓  Tipos de datos
#    Cambiamos los tipos de datos de las columnas a los más adecuados para optimizar el rendimiento.

df_baking['invoice'] = df_baking['invoice'].astype(str)       # Convertimos los códigos de factura a string (evita errores de agrupación)
df_baking['stockcode'] = df_baking['stockcode'].astype('category')
df_baking['description'] = df_baking['description'].astype('category')
df_baking['country'] = df_baking['country'].astype('category')
df_baking['invoicedate'] = pd.to_datetime(df_baking['invoicedate'])


# ✓ Eliminar facturas canceladas:
#   Las devoluciones tienen valores negativos en Quantity y normalmente un código de factura que empieza con 'C'.
#   Estas transacciones no son útiles para MBA, ya que no representan compras reales.
#   En este caso lo resolveremos, asegurandonos que solo se incluyan transacciones de compra (cantidad > 0)
df_baking = df_baking[df_baking['quantity'] > 0]


df = df_baking.copy()

# ✓ Revisar df
print("\n✅ Shape df:", df.shape)
print("\n✅ Muestra Aleatoria de Observaciones en df:\n")
display(df.sample(10,random_state=2025))
print("\n✅ Info general df:")
print(df.info())


✅ Shape df: (354345, 8)

✅ Muestra Aleatoria de Observaciones en df:



,invoice,stockcode,description,quantity,invoicedate,price,customer id,country
295211,562778,21175,gin + tonic diet metal sign,24,2011-08-09 12:55:00,2.5,"17,591.0",United Kingdom
250060,558995,22729,alarm clock bakelike orange,4,2011-07-05 12:03:00,3.8,"17,671.0",United Kingdom
393456,570827,22567,20 dolly pegs retrospot,2,2011-10-12 13:14:00,1.4,"15,831.0",United Kingdom
168213,551014,20728,lunch bag cars blue,10,2011-04-26 11:04:00,1.6,"13,209.0",United Kingdom
517195,580044,23090,vintage glass t-light holder,12,2011-12-01 12:36:00,0.8,"14,869.0",United Kingdom
355080,567905,21974,set of 36 paisley flower doilies,12,2011-09-22 16:37:00,1.4,"12,952.0",United Kingdom
224615,556538,22791,t-light glass fluted antique,12,2011-06-13 12:42:00,1.2,"15,845.0",United Kingdom
464731,576179,79191C,retro plastic elephant tray,144,2011-11-14 11:09:00,0.6,"13,694.0",United Kingdom
480810,577318,23238,set of 4 knick knack tins london,6,2011-11-18 13:42:00,4.2,"17,675.0",United Kingdom
465188,576213,16216,letter shape pencil sharpener,80,2011-11-14 12:47:00,0.1,"18,117.0",United Kingdom



✅ Info general df:
<class 'pandas.core.frame.DataFrame'>
Index: 354345 entries, 0 to 541893
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   invoice      354345 non-null  object        
 1   stockcode    354345 non-null  category      
 2   description  354345 non-null  category      
 3   quantity     354345 non-null  int64         
 4   invoicedate  354345 non-null  datetime64[ns]
 5   price        354345 non-null  float64       
 6   customer id  354345 non-null  float64       
 7   country      354345 non-null  category      
dtypes: category(3), datetime64[ns](1), float64(2), int64(1), object(1)
memory usage: 18.2+ MB
None


## ✔️ 5. EDA DESCRIPTIVO

In [6]:
display(df.describe(include='number').T)
display(df.describe(include='category').T)

,count,mean,std,min,25%,50%,75%,max
quantity,"354,345.0",12.0,190.4,1.0,2.0,4.0,12.0,"80,995.0"
price,"354,345.0",3.0,17.9,0.0,1.2,1.9,3.8,"8,142.8"
customer id,"354,345.0","15,552.4","1,594.5","12,346.0","14,194.0","15,522.0","16,931.0","18,287.0"


,count,unique,top,freq
stockcode,354345,3645,85123A,1947
description,354345,3833,white hanging heart t-light holder,1940
country,354345,1,United Kingdom,354345


## ✔️ 6. EDA VISUAL - TOP PRODUCTS

In [7]:
top_products = (df['description']
                .value_counts()
                .nlargest(10))


top_products_df = pd.DataFrame(top_products).reset_index()
top_products_df

,description,count
0,white hanging heart t-light holder,1940
1,jumbo bag red retrospot,1464
2,regency cakestand 3 tier,1426
3,assorted colour bird ornament,1333
4,party bunting,1308
5,lunch bag red retrospot,1147
6,lunch bag black skull.,1049
7,set of 3 cake tins pantry design,1020
8,paper chain kit 50's christmas,982
9,heart of wicker small,952


In [8]:

# ◯ Visualización de los productos más comprados
# ---------------------------------------------------------------------------------

# ✓ Visualización de los productos más comprados
#    Se utiliza un gráfico de barras horizontales para mostrar los 10 productos más comprados.
#    Se ordenan de mayor a menor frecuencia de compra.

# Datos: suma total por producto
product_counts = df.sum().sort_values(ascending=False).head(10)  # De mayor a menor

# Paleta de colores degradados de rosa (color fuerte arriba)
colors = sns.color_palette("husl", len(product_counts))  # Paleta de colores viridis

# Crear gráfico
plt.figure(figsize=(12, 6))
product_counts.sort_values().plot(kind='barh', color=colors)  # sort para que el más alto esté arriba
plt.title('Top 10 productos más comprados', fontsize=14)
plt.xlabel('Frecuencia de compra')
plt.ylabel('Producto')
plt.grid(axis='x', alpha=0.8)
plt.tight_layout()
plt.show()

TypeError: 'Categorical' with dtype category does not support reduction 'sum'

## ✔️ 7. CREAR LA MATRIZ DE TRANSACCIONES (Basket Matrix)

The basket matrix contendra la cantidad comprada de cada producto por transaccion (invoice).

Este data frame es básicamente la cesta que nuestros clientes llevan al cajero de nuestra tienda.  
Nos muestra cuánto compró este cliente/transacción (N.º de factura) por un artículo en particular.  
Si el número es 0, el cliente no compró ese artículo.  
Si muestra otro valor (8, por ejemplo), significa que el cliente ha comprado hasta 8 artículos.  

Para esto:  
Agrupamos los productos por factura y convertimos cada factura en una fila con los productos como columnas.

In [10]:
# ◯ Paso 1: Agrupar por factura y producto para construir la cesta
# ---------------------------------------------------------------------------------

# ✓ Sumamos las cantidades de cada producto dentro de cada factura
basket = df.groupby(['invoice', 'description'], observed=True)['quantity'].sum()
basket

invoice  description                        
536365   cream cupid hearts coat hanger          8
         glass star frosted t-light holder       6
         knitted union flag hot water bottle     6
         red woolly hottie white heart.          6
         set 7 babushka nesting boxes            2
                                                ..
581585   zinc willie winkie  candle stick       24
581586   doormat red retrospot                  10
         large cake stand  hanging strawbery     8
         red retrospot round cake tins          24
         set of 3 hanging owls ollie beak       24
Name: quantity, Length: 344293, dtype: int64

In [11]:
# ◯ Paso 2: Convertir la serie a una tabla con productos como columnas
# ---------------------------------------------------------------------------------

# ✓ Convertimos la serie a un DataFrame

basket = basket.unstack() # Unstack convierte las combinaciones (invoice, description) en una matriz
basket

description,10 colour spaceboy pen,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 hanging eggs hand painted,12 ivory rose peg place settings,12 message cards with envelopes,12 pencil small tube woodland,12 pencils small tube red retrospot,12 pencils small tube skull,...,zinc star t-light holder,zinc sweetheart soap dish,zinc sweetheart wire letter rack,zinc t-light holder star large,zinc t-light holder stars large,zinc t-light holder stars small,zinc top 2 door wooden shelf,zinc willie winkie candle stick,zinc wire kitchen organiser,zinc wire sweetheart letter tray
invoice,,,,,,,,,,,,,,,,,,,,,
536365,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536366,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536367,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536368,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536369,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581582,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
581583,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
581584,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# ◯ Paso 3: Reemplazar valores faltantes (productos no comprados) por 0
# ---------------------------------------------------------------------------------

# ✓ Reemplazamos los valores NaN por 0
#   Así sabemos qué productos no fueron comprados en cada factura

basket = basket.reset_index().fillna(0).set_index('invoice')
    #   reset_index() convierte el índice en una columna normal
    #   fillna(0) reemplaza los NaN por 0
    #   set_index('invoice') vuelve a establecer la columna 'invoice' como índice
basket

description,10 colour spaceboy pen,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 hanging eggs hand painted,12 ivory rose peg place settings,12 message cards with envelopes,12 pencil small tube woodland,12 pencils small tube red retrospot,12 pencils small tube skull,...,zinc star t-light holder,zinc sweetheart soap dish,zinc sweetheart wire letter rack,zinc t-light holder star large,zinc t-light holder stars large,zinc t-light holder stars small,zinc top 2 door wooden shelf,zinc willie winkie candle stick,zinc wire kitchen organiser,zinc wire sweetheart letter tray
invoice,,,,,,,,,,,,,,,,,,,,,
536365,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536366,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536367,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536368,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536369,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581582,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581583,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
581584,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### → ENCODE DE DATA:

En el market basket analysis, lo importante no es la cantidad comprada de cada artículo, sino que lo importante es saber si un artículo fue comprado o no.
Ya que nuestro fin es saber cuál es la asociación entre comprar ciertos artículos y comprar otros.
Por lo tanto, necesitamos codificar los datos de la canasta en datos binarios que indiquen si un artículo fue comprado (1) o no (0).

De esta forma, generamos un data frame que nos muestra si un artículo en particular fue comprado o no.

In [13]:
# Aplicamos map a cada columna para convertir los valores a 1 o 0

basket = basket.apply(lambda col: col.map(lambda x: 1 if x > 0 else 0))
basket

description,10 colour spaceboy pen,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 hanging eggs hand painted,12 ivory rose peg place settings,12 message cards with envelopes,12 pencil small tube woodland,12 pencils small tube red retrospot,12 pencils small tube skull,...,zinc star t-light holder,zinc sweetheart soap dish,zinc sweetheart wire letter rack,zinc t-light holder star large,zinc t-light holder stars large,zinc t-light holder stars small,zinc top 2 door wooden shelf,zinc willie winkie candle stick,zinc wire kitchen organiser,zinc wire sweetheart letter tray
invoice,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536369,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581582,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581584,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### → FILTRAR TRANSACCIONES (Solo compras con mas de un articulo)

En el Market Basket Analysis, vamos a descubrir la asociación entre 2 o más productos que se compran según los datos históricos.  
Por lo tanto, las transacciones que solo mueven un artículo, no nos seria de utilidad.   
Es decir, ¿cómo podríamos descubrir la asociación entre 2 o más productos si solo se compró 1 producto?  
Entonces, el siguiente paso es filtrar las transacciones que compraron más de 1 artículo.  

Según el resultado, podemos ver que hay 15,376 transacciones que compraron más de 1 artículo.  
Esto significa que el 92.35 % de los datos de la canasta son transacciones en las que se compraron más de 1 artículo.

In [14]:
basket = basket[(basket > 0).sum(axis=1) >= 2]

# ◯ Filtrar transacciones con al menos 2 productos
# ---------------------------------------------------------------------------------
# Paso 1: (basket > 0) crea un DataFrame booleano donde cada celda es True si ese producto fue comprado (> 0) y False si no.
# Paso 2: .sum(axis=1) suma los valores True por fila (es decir, cuenta cuántos productos fueron comprados en cada transacción).
# Paso 3: >= 2 mantiene solo las transacciones donde se compraron al menos 2 productos.
# Paso 4: Finalmente, se aplica ese filtro al DataFrame original 'basket', dejando solo las transacciones útiles para análisis de canasta
basket

description,10 colour spaceboy pen,12 coloured party balloons,12 daisy pegs in wood box,12 egg house painted wood,12 hanging eggs hand painted,12 ivory rose peg place settings,12 message cards with envelopes,12 pencil small tube woodland,12 pencils small tube red retrospot,12 pencils small tube skull,...,zinc star t-light holder,zinc sweetheart soap dish,zinc sweetheart wire letter rack,zinc t-light holder star large,zinc t-light holder stars large,zinc t-light holder stars small,zinc top 2 door wooden shelf,zinc willie winkie candle stick,zinc wire kitchen organiser,zinc wire sweetheart letter tray
invoice,,,,,,,,,,,,,,,,,,,,,
536365,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536366,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536367,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536368,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536372,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581582,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581583,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581584,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## ✔️ 8. APLICO EL ALGORITMO

Después de generar la cesta de productos en el formato necesario, es el momento de aplicar el algoritmo.  
Este algoritmo usa dos funciones que provienen de la libreria mlxtend:
1. **apriori:** Encontrar los conjuntos de productos frecuentes y 
2. **association_rules:** Generar las reglas de asociación entre esos productos.


### → Conjunto de productos Frecuentes (Funcion **Apriori**)

- Encuentra los conjuntos frecuentes en un conjunto de datos, es decir, grupos de artículos que se compran juntos con una frecuencia superior al valor de soporte especificado.
- Se utiliza para identificar qué combinaciones de artículos son compradas con mayor frecuencia.

Al aplicar el algoritmo Apriori, podemos definir los datos frecuentes que deseamos dando el valor de soporte.
En este caso, defino los artículos comprados con mayor frecuencia como aquellos que se compran al menos un 3% del total de las transacciones.
Esto significa que asignaré un valor de soporte de 0.03.
Después de eso, agregué otra columna llamada "length" que contiene el número de artículos que se compraron.

In [29]:
# ◯ Paso 1: Aplicamos el algoritmo Apriori para encontrar conjuntos de artículos frecuentes
 
basket = basket.astype(bool)  # Convertimos a booleano (True/False) para el algoritmo Apriori

frequent_itemsets = apriori(
    basket,                # ✓ DataFrame binario donde 1 indica que el artículo fue comprado y 0 que no
    min_support=0.03,      # ✓ Umbral mínimo de soporte: solo se consideran combinaciones presentes en al menos el 3% de las transacciones
    use_colnames=True      # ✓ Muestra los nombres reales de los artículos en lugar de índices
)

    
# ◯ Paso 2: Ordenamos los conjuntos frecuentes por soporte en orden descendente

frequent_itemsets = frequent_itemsets.sort_values(
    'support',             # ✓ Ordenamos en función del soporte (frecuencia relativa del conjunto)
    ascending=False        # ✓ De mayor a menor soporte
)
    

# ◯ Paso 3: Reiniciamos los índices del DataFrame ordenado

frequent_itemsets = frequent_itemsets.reset_index(drop=True)  # ✓ Eliminamos el índice anterior y creamos uno nuevo desde cero

# ◯ Paso 4: Agregamos una nueva columna que indica la cantidad de artículos en cada conjunto frecuente

frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(
    lambda x: len(x)       # ✓ Contamos cuántos artículos hay en cada conjunto de ítems
)

# ◯ Mostramos el DataFrame final con:
# ✓ Conjuntos frecuentes de artículos
# ✓ Su soporte (frecuencia)
# ✓ Y la cantidad de artículos en cada conjunto
frequent_itemsets

,support,itemsets,length
0,0.12,(white hanging heart t-light holder),1
1,0.09,(jumbo bag red retrospot),1
2,0.09,(regency cakestand 3 tier),1
3,0.08,(assorted colour bird ornament),1
4,0.08,(party bunting),1
...,...,...,...
103,0.03,(doormat union flag),1
104,0.03,"(lunch bag pink polkadot, lunch bag red retros...",2
105,0.03,(set of 3 heart cookie cutters),1
106,0.03,(dolly girl lunch box),1


Como puedes ver, hay 108 transacciones que se consideran como artículos comprados con mayor frecuencia.  
El "White Hanging Heart T-Light Holder" es el artículo más comprado con un valor de soporte de 12.1%.  
Esto significa que el artículo se compró 1866 veces de todas las transacciones.

Si solo queremos los conjuntos de 2 ítems que aparecen en al menos el 3% de las transacciones:

In [31]:
frequent_itemsets[ 
    (frequent_itemsets['length']  == 2) &       # ✓ Conjuntos que tienen exactamente 2 artículos
    (frequent_itemsets['support'] >= 0.03)      # ✓ Cuyo soporte (frecuencia) es mayor o igual a 3%
]

,support,itemsets,length
82,0.03,"(jumbo bag red retrospot, jumbo bag pink polka...",2
92,0.03,"(lunch bag red retrospot, lunch bag black sku...",2
100,0.03,"(roses regency teacup and saucer, green regenc...",2
104,0.03,"(lunch bag pink polkadot, lunch bag red retros...",2


###  → Encontrar Reglas de Asociación entre los productos (Funcion **association rules**) 
- Genera reglas de asociación a partir de los conjuntos frecuentes obtenidos con el algoritmo Apriori.
- Estas reglas muestran las relaciones entre los productos, por ejemplo, si un cliente compra un producto A, ¿qué tan probable es que compre también el producto B?
- Se pueden filtrar las reglas utilizando métricas como confianza, soporte y lift para encontrar las relaciones más significativas.

Cada regla se evalúa con 3 métricas:

* Soporte (support): frecuencia de la combinación en todas las transacciones  
    - Soporte({A,B}) = ocurrencias de A y B juntos / total transacciones

* Confianza (confidence): probabilidad de que B ocurra dado que ocurrió A  
    - Confianza(A => B) = Soporte({A,B}) / Soporte({A})

* Elevación (lift): mide si la aparición de B está realmente relacionada con A  
    - Lift(A => B) = Confianza(A => B) / Soporte(B)  
Si:  
    - Lift > 1: A y B están positivamente relacionados  
    - Lift = 1: A y B son independientes  
    - Lift < 1: A y B tienen relación negativa  

**Metricas:**
* **support**:      Frecuencia conjunta del antecedente y consecuente
* **confidence**:   Probabilidad de que ocurra el consecuente dado el antecedente
* **lift**: 	    Relación entre A y B: si es > 1 hay asociación positiva, si < 1 es débil

Después de aplicar el algoritmo Apriori y encontrar los artículos que se compran con frecuencia, ahora es momento de aplicar las reglas de asociación.
A partir de estas reglas, podemos extraer información e incluso descubrir conocimientos sobre qué artículos son más efectivos para venderse juntos.
Ese es el objetivo principal de este proyecto.

In [32]:
rules = association_rules(
    frequent_itemsets,     # ✓ Conjuntos frecuentes generados previamente con Apriori
    metric='lift',         # ✓ Métrica principal que se va a evaluar para filtrar reglas
    min_threshold=1        # ✓ Se seleccionan solo reglas con lift >= 1 (asociación no trivial)
).sort_values('lift', ascending=False)  # ✓ Se ordenan de mayor a menor lift

rules = rules.reset_index(drop=True)   # ✓ Se reinicia el índice del DataFrame resultante

rules                                   # ✓ Se muestra el DataFrame de reglas generadas


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(green regency teacup and saucer),(roses regency teacup and saucer),0.04,0.04,0.03,0.78,17.72,1.00,0.03,4.30,0.98,0.59,0.77,0.74
1,(roses regency teacup and saucer),(green regency teacup and saucer),0.04,0.04,0.03,0.71,17.72,1.00,0.03,3.26,0.99,0.59,0.69,0.74
2,(lunch bag red retrospot),(lunch bag pink polkadot),0.07,0.06,0.03,0.42,7.63,1.00,0.03,1.63,0.94,0.31,0.39,0.49
3,(lunch bag pink polkadot),(lunch bag red retrospot),0.06,0.07,0.03,0.56,7.63,1.00,0.03,2.09,0.92,0.31,0.52,0.49
4,(jumbo bag pink polkadot),(jumbo bag red retrospot),0.05,0.09,0.03,0.62,6.70,1.00,0.03,2.42,0.90,0.29,0.59,0.49
5,(jumbo bag red retrospot),(jumbo bag pink polkadot),0.09,0.05,0.03,0.35,6.70,1.00,0.03,1.46,0.94,0.29,0.32,0.49
6,(lunch bag black skull.),(lunch bag red retrospot),0.06,0.07,0.03,0.49,6.68,1.00,0.03,1.81,0.91,0.30,0.45,0.46
7,(lunch bag red retrospot),(lunch bag black skull.),0.07,0.06,0.03,0.43,6.68,1.00,0.03,1.65,0.92,0.30,0.39,0.46


1. **Lift (Highest Association):**  
    From the association_rules results above, we could see that  
    * ROSES R REGENCY TEACUP AND SAUCER and 
    * GREEN REGENCY TEACUP AND SAUCE

    are the items that has the **highest association** each other since these two items has the highest **“lift” value**.  
    **The higher the lift value, the higher the association between the items willl.**   
    **If the lift value is more than 1, it is enough for us to say that those two items are associated each other.**   
    In thise case, the highest value is 17.717 which is very high. It means these 2 items are very good to be sold together.

2. **Support:**
    Beside that, we could also see the **support value** of 
    * ROSES REGENCY TEACUP AND SAUCER and 
    * GREEN REGENCY TEACUP AND SAUCER 

    are 0.0309% which means there are 3.09% out of total transaction that these 2 items were sold together.  
    In number, it is 476 times.

3. **Confidence:**
    From the confidence, we could even extract more information. 
    Remember that **the confidence value is influenced by the antecedent and consequent.**   
    If the antecedent is higher than the consequent, then the rule that will be applied is rule number 1 (not number 2). vice versa.   
    In this case, the antecedent value is higher than the consequent value. It means we will apply rule number 1 which is 𝐺𝑅𝐸𝐸𝑁 𝑅𝐸𝐺𝐸𝑁𝐶𝑌 𝑇𝐸𝐴𝐶𝑈𝑃 𝐴𝑁𝐷 𝑆𝐴𝑈𝐶𝐸𝑅 → 𝑅𝑂𝑆𝐸𝑆 𝑅𝐸𝐺𝐸𝑁𝐶𝑌 𝑇𝐸𝐴𝐶𝑈𝑃 𝐴𝑁𝐷 𝑆𝐴𝑈𝐶𝐸.  

    In a more detail explanation, it means that a customer will tends to bought Roses Regency Teacup and Saucer AFTER they bought Green Regency Teacup And Saucer.   
    Not in the other way around.   
    This could be a very valuable information, because we are now aware which products should we put the discounts on.   
    We could give a discounts on Roses Regency Teacup and Sauce if a customer buy Green Regency Teacup and Saucer.


###  → Mostrar Reglas Legibles
Definimos una funcion que muestre las reglas de asociación en formato legible.  
Convierte sets en texto y redondea columnas clave si existen.

In [34]:
pd.options.display.float_format = '{:.2f}'.format

def mostrar_reglas_legibles(rules_df):
    """
    Muestra reglas de asociación en formato legible.
    Convierte sets en texto y redondea columnas clave si existen.
    """
    df = rules_df.copy()
    
    # Convertir conjuntos a strings legibles
    df['antecedents'] = df['antecedents'].apply(lambda x: ', '.join(sorted(list(x))))
    df['consequents'] = df['consequents'].apply(lambda x: ', '.join(sorted(list(x))))

    # Columnas clave (si existen)
    columnas = ['support', 'confidence', 'lift']
    columnas_existentes = [col for col in columnas if col in df.columns]
    
    # Redondear columnas numéricas existentes
    df[columnas_existentes] = df[columnas_existentes].round(4)

    # Formatear support como porcentaje con 1 decimal
    if 'support' in columnas_existentes:
        df['support'] = (df['support'] * 100).map('{:.2f}%'.format)

    return df[['antecedents', 'consequents'] + columnas_existentes]

# Mostrar reglas legibles
rules_legibles = mostrar_reglas_legibles(rules)
rules_legibles

,antecedents,consequents,support,confidence,lift
0,green regency teacup and saucer,roses regency teacup and saucer,3.10%,0.78,17.72
1,roses regency teacup and saucer,green regency teacup and saucer,3.10%,0.71,17.72
2,lunch bag red retrospot,lunch bag pink polkadot,3.06%,0.42,7.63
3,lunch bag pink polkadot,lunch bag red retrospot,3.06%,0.56,7.63
4,jumbo bag pink polkadot,jumbo bag red retrospot,3.29%,0.62,6.70
5,jumbo bag red retrospot,jumbo bag pink polkadot,3.29%,0.35,6.70
6,lunch bag black skull.,lunch bag red retrospot,3.15%,0.49,6.68
7,lunch bag red retrospot,lunch bag black skull.,3.15%,0.43,6.68


## ✔️ 9. Conclusions

In this articles, we’ve done a Market Basket Analysis using an actual online retail transaction data from UK.  
The result of this market basket analysis could be used for a data-driven marketing strategy and decision making. 
In this datasets, we could generates several business insights as follows :

1. **Item Placements:**  
    We could put 
    * ROSES REGENCY TEACUP AND SAUCER and 
    * GREEN REGENCY TEACUP AND SAUCER   
    in a closer place, maybe in a same shelf or any other closer place.

2. **Products Bundling:**  
    We could put 
    * ROSES REGENCY TEACUP AND SAUCER and 
    * GREEN REGENCY TEACUP AND SAUCER 
    as a single bundle of product with a lower price compare to each price combined.   
    This way will attract more sales and generates more income.

3. **Customer Recommendation and Discounts:**  
    We could put Roses Regency Teacup and Saucer in the cashier, so that every time a customer bought Green Regency Teacup and Saucer,   
    we could offer and recommend them to buy Roses Regency Teacup and Saucer with a lower price.




Sources :

    Halim, Octavia, and Alianto. 2019. Designing Facility Layout of an Amusement Arcade using Market Basket Analysis. Procedia Computer Science, Vol 161, Page 623–629.
    (https://www.sciencedirect.com/science/article/pii/S1877050919318769)

    Maitra, Sarit. 2019. Association Rule Mining using Market Basket Analysis.
    (https://towardsdatascience.com/market-basket-analysis-knowledge-discovery-in-database-simplistic-approach-dc41659e1558)

    Subramanian, Dhilip. 2019. Association Discovery — the Apriori Algorithm.
    (https://medium.com/towards-artificial-intelligence/association-discovery-the-apriori-algorithm-28c1e71e0f04)

    Chauhan, Nagesh Singh. 2019. Market Basket Analysis.
    (https://towardsdatascience.com/market-basket-analysis-978ac064d8c6)

    Li, Susan. 2017. A Gentle Introduction on Market Basket Analysis — Association Rules.
    (https://towardsdatascience.com/a-gentle-introduction-on-market-basket-analysis-association-rules-fa4b986a40ce)

    https://archive.ics.uci.edu/ml/datasets/Online+Retail+II



## ✔️ 10. GUARDAR RESULTADOS

In [36]:
# ◯ GUARDAR RESULTADOS
rules_legibles.to_csv('association_rules_final.csv', index=False)
print("\n✅ Reglas exportadas como 'association_rules_final.csv'")


✅ Reglas exportadas como 'association_rules_final.csv'


# New Process

In [19]:
# ◯ 0. CONFIGURACIÓN INICIAL

# ✓ Ruta del dataset
ruta_archivo = 'ruta/del/archivo.csv'  # <-- Cambia esto

# ✓ Parámetros de EDA
top_n_productos = 10  # Número de productos más vendidos a graficar

# ✓ Parámetros de Apriori
min_support = 0.01    # Soporte mínimo
min_lift = 1.0        # Lift mínimo
min_confidence = 0.5  # Confianza mínima (opcional si quieres filtrar más adelante)

# ✓ Opciones de limpieza
eliminar_facturas_canceladas = True  # True = sí eliminar las facturas que empiezan con "C"

# ----------------------------------------
# ◯ 1. IMPORTAR LIBRERÍAS
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules

# ◯ 2. CARGA DE DATOS
data = pd.read_csv(ruta_archivo)

# ◯ 3. EDA INICIAL
print("\n✅ Shape inicial:", data.shape)
print("\n✅ Primeras filas:\n", data.head())
print("\n✅ Info general:")
print(data.info())
print("\n✅ Valores nulos por columna:\n", data.isnull().sum())

# ◯ 4. LIMPIEZA
df = data.dropna()
df.columns = df.columns.str.lower()

# ✓ Filtrar United Kingdom
df = df[df['country'] == 'United Kingdom'].copy()

# ✓ Tipos de datos
df['invoice'] = df['invoice'].astype('category')
df['stockcode'] = df['stockcode'].astype('category')
df['description'] = df['description'].astype('category')
df['country'] = df['country'].astype('category')
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

# ✓ Opcional: eliminar facturas canceladas
if eliminar_facturas_canceladas:
    df = df[~df['invoice'].str.startswith('C')]

# ◯ 5. EDA Visual - Top productos
top_products = (df['description']
                .value_counts()
                .nlargest(top_n_productos))

plt.figure(figsize=(10,6))
sns.barplot(x=top_products.values, y=top_products.index, palette='viridis')
plt.title(f'Top {top_n_productos} Productos Más Vendidos en United Kingdom')
plt.xlabel('Cantidad Vendida')
plt.ylabel('Producto')
plt.show()

# ◯ 6. CREAR MATRIZ TIPO CESTA (ver siguiente chank para detectar opciones)
basket = (df
          .groupby(['invoice', 'description'])['quantity']
          .sum().unstack().fillna(0)
          .applymap(lambda x: 1 if x > 0 else 0))

# ◯ 7. APLICAR ALGORITMO APRIORI
frequent_itemsets = apriori(basket, min_support=min_support, use_colnames=True)

print(f"\n✅ Frecuencias encontradas (con min_support={min_support}):")
print(frequent_itemsets.head())

# ◯ 8. EXTRAER REGLAS DE ASOCIACIÓN
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=min_lift)

# ✓ Opcional: filtrar por confianza
rules = rules[rules['confidence'] >= min_confidence]

# ✓ Ordenar por lift descendente
rules_sorted = rules.sort_values(by='lift', ascending=False)

print("\n✅ Reglas de asociación encontradas:")
print(rules_sorted.head())

# ◯ 9. GUARDAR RESULTADOS
rules_sorted.to_csv('association_rules_final.csv', index=False)
print("\n✅ Reglas exportadas como 'association_rules_final.csv'")


FileNotFoundError: [Errno 2] No such file or directory: 'ruta/del/archivo.csv'

In [ ]:
# ◯ PASO: Preparar la 'basket' para Apriori
# ===========================================

# ✓ Opción 1: Método clásico (groupby + unstack + applymap)
# ----------------------------------------------------------
# ◯ Pros:
#     ✓ Muy explícito: primero agrupa, luego pivotea y convierte.
#     ✓ Más control sobre sumar cantidades y luego binarizar (>0 ➔ 1)
# ◯ Contras:
#     ✗ Ligeramente más largo de escribir.

basket = (df_baking
          .groupby(['invoice', 'description'])['quantity']
          .sum()
          .unstack()
          .fillna(0)
          .applymap(lambda x: 1 if x > 0 else 0)
         )


# ✓ Opción 2: Método moderno (get_dummies)
# ----------------------------------------------------------
# ◯ Pros:
#     ✓ Código más corto y elegante.
#     ✓ Aprovecha automáticamente que get_dummies trabaja a 0/1.
# ◯ Contras:
#     ✗ Menos explícito si necesitas controlar cantidades exactas.

# --- Descomentar estas líneas para usar get_dummies:
# basket = (pd.get_dummies(df_baking.set_index('invoice')['description'])
#              .groupby('invoice')
#              .max()
#          )

# ===========================================
# ◯ Resultado: 
#     - Cada fila representa una factura (invoice)
#     - Cada columna representa un producto (description)
#     - 1 si el producto fue comprado en la factura, 0 si no.
# ===========================================

# ◯ Visualizar primeras filas
basket.head()
